# Schema Validation & Structured Outputs with Pydantic

This notebook covers how to force an LLM to output structured data that conforms to a specific schema defined using **Pydantic**. We will cover:
1. **Pydantic Schemas**: Defining models with validation rules.
2. **Structured Output**: Using `.with_structured_output` in LangChain.
3. **Complex Validation**: Leveraging Pydantic validators to check or clean the parsed output.

## Setup & Installation

Ensure you have the required packages installed and your API keys configured.

In [ ]:
# !pip install langchain langchain-openai pydantic python-dotenv

In [ ]:
import os

# 1. Try to load local .env if it exists
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# 2. Try to load Colab secrets if running in Google Colab
if "OPENAI_API_KEY" not in os.environ:
    try:
        from google.colab import userdata
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except ImportError:
        pass

# 3. Fallback to manual entry if not found
if "OPENAI_API_KEY" not in os.environ:
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 1. Defining a Pydantic Schema

Let's define a schema for a book recommendation.

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field

class BookRecommendation(BaseModel):
    title: str = Field(description="The title of the book")
    author: str = Field(description="The author of the book")
    genre: List[str] = Field(description="The genres of the book, up to 3")
    rating: float = Field(description="Expected rating out of 5 stars (e.g. 4.5)")
    summary: str = Field(description="A one-sentence summary of why the user would like it")
    target_audience: Optional[str] = Field(None, description="Optional description of the target reader")

## 2. Requesting Structured Output from LLM

Using LangChain's `.with_structured_output()` binds our Pydantic model directly to the LLM backend.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_llm = model.with_structured_output(BookRecommendation)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful librarian. Recommend a book based on the user's preference."),
    ("user", "{user_preference}")
])

chain = prompt | structured_llm

recommendation = chain.invoke({
    "user_preference": "I want a mind-bending sci-fi book that deals with time travel and has a dark tone."
})

print(type(recommendation))
print(recommendation.model_dump_json(indent=2))

## 3. Adding Pydantic Custom Validation Rules

We can add custom validation to Pydantic schemas using the `@field_validator` decorator to guarantee specific data formats (e.g. capitalizing names or enforcing numeric bounds).

In [ ]:
from pydantic import field_validator

class CustomBookRecommendation(BookRecommendation):
    @field_validator('rating')
    @classmethod
    def val_rating(cls, v: float) -> float:
        if v < 0 or v > 5:
            raise ValueError("Rating must be between 0 and 5")
        return v